## 0. Environment setup

Clone the thinking-budget vLLM patch and install runtime dependencies (`vllm`, `datasets`, `openai`, `pandas`); `torchaudio` is uninstalled since it is unused and otherwise conflicts with the pinned `torch` version on this image.

In [1]:
!git clone https://github.com/phishingupstream/vllm-thinking-budget.git
!pip install -q vllm datasets openai pandas
!pip uninstall -y -q torchaudio

Cloning into 'vllm-thinking-budget'...
remote: Enumerating objects: 36, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 36 (delta 15), reused 31 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (36/36), 168.05 KiB | 2.37 MiB/s, done.
Resolving deltas: 100% (15/15), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.0/316.0 MB 5.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 59.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 76.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 3.0 MB/s

### `VLLMServer` — manages the vLLM subprocess

Applies the `-1` placeholder patch needed for async scheduling on newer vLLM, writes the server config (including the thinking-budget logits processor), and starts/stops/health-checks the subprocess. Kept as-is from the original setup.

In [38]:
import os
import subprocess
import time
import urllib.request
import urllib.error


class VLLMServer:
    def __init__(
        self,
        model,
        port=8001,
        repo_dir="/kaggle/working/vllm-thinking-budget",
        work_dir="/kaggle/working",
    ):
        self.model = model
        self.port = port
        self.repo_dir = repo_dir
        self.work_dir = work_dir
        self.process = None

    def apply_patch(self):
        """Fixes the -1 placeholder bug (async scheduling in newer vLLM)."""

        print("[VLLMServer] Checking patch...", flush=True)

        path = os.path.join(
            self.repo_dir,
            "thinking_budget_processor.py"
        )

        if not os.path.exists(path):
            print(
                f"[VLLMServer] ERROR: Patch file not found: {path}",
                flush=True
            )
            raise FileNotFoundError(path)

        with open(path, "r") as f:
            content = f.read()

        old = (
            "        for i in range(self.last_scanned, end):\n"
            "            tid = tokens[i]"
        )

        new = """        for i in range(self.last_scanned, end):
            tid = tokens[i]
            if tid == -1:
                end = i
                break"""

        if old in content and "if tid == -1:" not in content:

            content = content.replace(old, new)

            with open(path, "w") as f:
                f.write(content)

            print("[VLLMServer] ✓ Patch applied.", flush=True)

        elif "if tid == -1:" in content:

            print("[VLLMServer] ✓ Patch already applied.", flush=True)

        else:

            print(
                "[VLLMServer] ⚠ WARNING: patch pattern not found.",
                flush=True
            )

    

    def write_config(self):

        print("[VLLMServer] Writing vLLM config...", flush=True)

        config = (
            f"model: {self.model}\n"
            "max-model-len: 16384\n"
            "tensor-parallel-size: 2\n"
            f"port: {self.port}\n"
            "logits-processors:\n"
            '- "thinking_budget_processor:ThinkingBudgetLogitsProcessor"\n'
        )

        self.config_path = os.path.join(
            self.work_dir,
            "vllm-config.yaml"
        )

        with open(self.config_path, "w") as f:
            f.write(config)

        print(
            f"[VLLMServer] ✓ Config written: {self.config_path}",
            flush=True
        )

        print(
            f"[VLLMServer] Model: {self.model}",
            flush=True
        )

        print(
            f"[VLLMServer] Port: {self.port}",
            flush=True
        )

        print(
            "[VLLMServer] Tensor parallel size: 2",
            flush=True
        )



    def start(self, timeout=360):

        print("\n" + "=" * 60, flush=True)
        print("[VLLMServer] Starting vLLM...", flush=True)
        print("=" * 60, flush=True)

        print("[VLLMServer] Killing old vLLM processes...", flush=True)

        subprocess.run(
            ["pkill", "-9", "-f", "vllm"],
            check=False
        )

        subprocess.run(
            ["pkill", "-9", "-f", "VLLM::"],
            check=False
        )

        time.sleep(3)

        print("[VLLMServer] Old processes cleared.", flush=True)

        env = os.environ.copy()
        env["PYTHONPATH"] = (
            self.repo_dir
            + ":"
            + env.get("PYTHONPATH", "")
        )

        log_path = os.path.join(
            self.work_dir,
            "vllm.log"
        )

        print(
            f"[VLLMServer] Log file: {log_path}",
            flush=True
        )

        log_file = open(log_path, "w")

        print("[VLLMServer] Launching vLLM process...", flush=True)

        self.process = subprocess.Popen(
            [
                "vllm",
                "serve",
                self.model,
                "--config",
                self.config_path,
            ],
            env=env,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )

        print(
            f"[VLLMServer] ✓ Process started. PID={self.process.pid}",
            flush=True
        )

        print(
            f"[VLLMServer] Waiting for http://localhost:{self.port}/health",
            flush=True
        )

        start_time = time.time()
        last_log_check = 0

        for attempt in range(timeout // 2):

            elapsed = time.time() - start_time

            # --------------------------------------------------
            # Check whether the process is still alive
            # --------------------------------------------------
            if self.process.poll() is not None:

                return_code = self.process.returncode

                print(
                    "\n[VLLMServer] ❌ vLLM process exited!",
                    flush=True
                )

                print(
                    f"[VLLMServer] Return code: {return_code}",
                    flush=True
                )

                print(
                    f"[VLLMServer] Check log: {log_path}",
                    flush=True
                )

                # Show last lines of log
                try:
                    with open(log_path, "r") as f:
                        lines = f.readlines()

                    print("\n----- LAST 30 LOG LINES -----", flush=True)

                    for line in lines[-30:]:
                        print(line.rstrip(), flush=True)

                    print("----- END LOG -----\n", flush=True)

                except Exception as e:
                    print(
                        f"[VLLMServer] Could not read log: {e}",
                        flush=True
                    )

                raise RuntimeError(
                    f"vLLM exited with return code {return_code}"
                )

            # --------------------------------------------------
            # Check health endpoint
            # --------------------------------------------------
            try:

                response = urllib.request.urlopen(
                    f"http://localhost:{self.port}/health",
                    timeout=2
                )

                status = response.status

                if status == 200:

                    elapsed = time.time() - start_time

                    print(
                        "\n" + "=" * 60,
                        flush=True
                    )

                    print(
                        "[VLLMServer] ✓ SERVER READY!",
                        flush=True
                    )

                    print(
                        f"[VLLMServer] Startup time: {elapsed:.1f}s",
                        flush=True
                    )

                    print(
                        f"[VLLMServer] URL: http://localhost:{self.port}/v1",
                        flush=True
                    )

                    print(
                        "=" * 60,
                        flush=True
                    )

                    return

            except (
                urllib.error.URLError,
                urllib.error.HTTPError,
                ConnectionError,
                TimeoutError,
            ):
                pass

            # --------------------------------------------------
            # Print progress every 10 seconds
            # --------------------------------------------------
            if elapsed - last_log_check >= 10:

                print(
                    f"[VLLMServer] Still starting... "
                    f"{elapsed:.0f}s / {timeout}s",
                    flush=True
                )

                # Read last few lines from vLLM log
                try:

                    with open(log_path, "r") as f:
                        lines = f.readlines()

                    if lines:

                        print(
                            "[VLLMServer] Latest log:",
                            flush=True
                        )

                        for line in lines[-3:]:
                            print(
                                "   " + line.rstrip(),
                                flush=True
                            )

                except Exception:
                    pass

                last_log_check = elapsed

            time.sleep(2)

        # ------------------------------------------------------
        # Timeout
        # ------------------------------------------------------

        print(
            "\n[VLLMServer] ❌ TIMEOUT!",
            flush=True
        )

        print(
            f"[VLLMServer] Server was not ready after {timeout}s.",
            flush=True
        )

        print(
            f"[VLLMServer] Full log: {log_path}",
            flush=True
        )

        raise RuntimeError(
            f"Server not ready after {timeout}s — "
            f"check {log_path}"
        )



    def stop(self):

        print("[VLLMServer] Stopping vLLM...", flush=True)

        subprocess.run(
            ["pkill", "-9", "-f", "vllm"],
            check=False
        )

        subprocess.run(
            ["pkill", "-9", "-f", "VLLM::"],
            check=False
        )

        print("[VLLMServer] ✓ vLLM stopped.", flush=True)

    def setup_and_start(self):

        print("\n" + "=" * 60, flush=True)
        print("[VLLMServer] SETUP START", flush=True)
        print("=" * 60, flush=True)

        self.apply_patch()

        self.write_config()

        self.start()

        print(
            "[VLLMServer] ✓ SETUP COMPLETE",
            flush=True
        )


## 1. Server setup (thinking-budget setup + its patch)


In [8]:
Model = "Qwen/Qwen3-8B"
server = VLLMServer(model=Model, port=8001)
server.setup_and_start()



[VLLMServer] SETUP START
[VLLMServer] Checking patch...
[VLLMServer] ✓ Patch already applied.
[VLLMServer] Writing vLLM config...
[VLLMServer] ✓ Config written: /kaggle/working/vllm-config.yaml
[VLLMServer] Model: Qwen/Qwen3-8B
[VLLMServer] Port: 8001
[VLLMServer] Tensor parallel size: 2

[VLLMServer] Starting vLLM...
[VLLMServer] Killing old vLLM processes...
[VLLMServer] Old processes cleared.
[VLLMServer] Log file: /kaggle/working/vllm.log
[VLLMServer] Launching vLLM process...
[VLLMServer] ✓ Process started. PID=253
[VLLMServer] Waiting for http://localhost:8001/health
[VLLMServer] Still starting... 10s / 360s
[VLLMServer] Still starting... 20s / 360s
[VLLMServer] Still starting... 30s / 360s
[VLLMServer] Latest log:
   (APIServer pid=253) INFO 09-12 21:35:12 [api_utils.py:347]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
   (APIServer pid=253) INFO 09-12 21:35:12 [api_utils.py:347]
   (APIServer pid=253) INFO 09-12 21:35:12 [api_utils.py:286] non-default args: {'model_tag': 'Qwen/Qwen3-8B', 'port'

## 2. Benchmark definitions with labels (benchmark, difficulty_tier, task_type)

Nine benchmarks across the three tiers from the target table — 3 per tier, one per task-type column (math / general knowledge / instruction-following for easy; math / general knowledge / agentic-tool-use for medium; math / PhD-level science / coding for hard). Each `load()` returns `{"question": ..., "gold": ...}` and each `grade()` scores one output — this label is stored with every question from the moment it's collected, so disaggregation later needs no re-running. See inline NOTE comments for benchmarks whose official grader is reimplemented as a simplified approximation (IFEval, BFCL v3, LiveCodeBench).

### **you should login to your HF account, Put the token in the kaggle secrets, Then Visit <https://huggingface.co/datasets/Idavidrein/gpqa> to accept dataset terms**

In [86]:
import re
import ast
import json
import glob
import random
import zlib
import base64
import subprocess
import tempfile
import os
from abc import ABC, abstractmethod
from datasets import load_dataset
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF")

class Benchmark(ABC):
    """Base class — implement load() and grade() for any new benchmark.

    Every subclass declares:
      - name: str, unique identifier used in the raw results table
      - difficulty_tier: "easy" | "medium" | "hard"
      - task_type: category label used for disaggregation (matches the target
        benchmark table: math / general_knowledge / instruction_following /
        agentic_tool_use / phd_science / coding)
    """

    name = None
    difficulty_tier = None
    task_type = None

    @abstractmethod
    def load(self, n):
        """Returns list of {"question": str, "gold": any}"""
        ...

    @abstractmethod
    def grade(self, output_text, gold):
        """Returns True/False"""
        ...


def _extract_boxed(text):
    """Extract the last \\boxed{...} contents, handling nested braces."""
    idxs = [m.start() for m in re.finditer(r"\\boxed\{", text)]
    if not idxs:
        return None
    start = idxs[-1] + len("\\boxed{")
    depth = 1
    i = start
    while i < len(text) and depth > 0:
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
        i += 1
    return text[start:i - 1].strip()


# ================================================================== easy tier
class GSM8KBenchmark(Benchmark):
    """Easy / Math — GSM8K (documented Qwen3-8B thinking-mode score: 97.4%)."""

    name = "GSM8K"
    difficulty_tier = "easy"
    task_type = "math"

    def load(self, n=30):
        ds = load_dataset(
            "openai/gsm8k",
            "main",
            split="test").select(
            range(n))
        questions = []
        for row in ds:
            gold = re.search(
                r"####\s*(-?[\d,]+\.?\d*)",
                row["answer"]).group(1).replace(
                ",",
                "")
            questions.append({"question": row["question"], "gold": gold})
        return questions

    def grade(self, output_text, gold):
        numbers = re.findall(r"-?\d[\d,]*\.?\d*", output_text)
        if not numbers:
            return False
        try:
            return abs(
                float(numbers[-1].replace(",", "")) - float(gold)) < 1e-4
        except ValueError:
            return False


class MMLURedux(Benchmark):
    """Easy / General knowledge — MMLU-Redux (documented: 87.5%).

    MMLU-Redux re-annotates a subset of the original MMLU test sets to fix
    labeling errors, so it shares MMLU's per-subject schema. We mix a handful
    of subjects (as with the original MMLU-mix) for a representative sample.
    """

    name = "MMLU-Redux"
    difficulty_tier = "easy"
    task_type = "general_knowledge"

    DEFAULT_SUBJECTS = [
        "high_school_geography",
        "high_school_world_history",
        "miscellaneous",
        "professional_law",
    ]

    def __init__(self, subjects=None):
        self.subjects = subjects or self.DEFAULT_SUBJECTS

    def load(self, n=30):
        per_subject = max(1, n // len(self.subjects))
        questions = []
        for subject in self.subjects:
            ds = load_dataset(
                "edinburgh-dawg/mmlu-redux-2.0",
                subject,
                split="test")
            ds = ds.select(range(min(per_subject, len(ds))))
            for row in ds:
                choices = "\n".join(
                    f"{chr(65 + i)}. {c}"
                    for i, c in enumerate(row["choices"])
                )
                q = (
                    f"{row['question']}\n{choices}\n"
                    "Answer with just the letter."
                )
                questions.append(
                    {"question": q, "gold": chr(65 + row["answer"])})
        return questions[:n]

    def grade(self, output_text, gold):
        match = re.search(r"\b([A-D])\b", output_text.strip())
        return match is not None and match.group(1) == gold


class IFEvalBenchmark(Benchmark):
    """Easy / Instruction-following — IFEval (documented: 85.0%).

    NOTE (simplification): the official IFEval grader implements a full
    registry of ~25 verifiable-instruction checkers (instructions_registry.py
    in google-research/instruction_following_eval). Re-implementing all of
    them is out of scope here; this loader keeps only the subset of
    instruction_ids we implement a checker for (a handful of the most common
    ones: keyword existence/frequency, forbidden words, number of words/
    sentences/paragraphs, lowercase, bullet-list count). Swap in the official
    `instructions_registry` for publication-grade rigor.
    """

    name = "IFEval"
    difficulty_tier = "easy"
    task_type = "instruction_following"

    _SUPPORTED_PREFIXES = (
        "keywords:existence",
        "keywords:frequency",
        "keywords:forbidden_words",
        "length_constraints:number_words",
        "length_constraints:number_sentences",
        "length_constraints:number_paragraphs",
        "change_case:english_lowercase",
        "detectable_format:number_bullet_lists",
    )

    def load(self, n=30):
        ds = load_dataset("google/IFEval", split="train")
        questions = []
        for row in ds:
            ids = row["instruction_id_list"]
            if not any(i.startswith(self._SUPPORTED_PREFIXES) for i in ids):
                continue
            questions.append({
                "question": row["prompt"],
                "gold": {"instruction_id_list": ids, "kwargs": row["kwargs"]},
            })
            if len(questions) >= n:
                break
        return questions

    def _check_one(self, text, instr_id, kwargs):
        try:
            if instr_id.startswith("keywords:existence"):
                return all(kw.lower() in text.lower()
                           for kw in kwargs.get("keywords", []))
            if instr_id.startswith("keywords:frequency"):
                kw = kwargs.get("keyword", "")
                target = kwargs.get("frequency", 0)
                rel = kwargs.get("relation", "at least")
                count = text.lower().count(kw.lower())
                if rel == "at least":
                    return count >= target
                return count <= target
            if instr_id.startswith("keywords:forbidden_words"):
                return all(kw.lower() not in text.lower()
                           for kw in kwargs.get("forbidden_words", []))
            if instr_id.startswith("length_constraints:number_words"):
                n_words = len(text.split())
                target = kwargs.get("num_words", 0)
                rel = kwargs.get("relation", "at least")
                if rel == "at least":
                    return n_words >= target
                return n_words <= target
            if instr_id.startswith("length_constraints:number_sentences"):
                n_sent = len(re.findall(r"[.!?]+", text))
                target = kwargs.get("num_sentences", 0)
                rel = kwargs.get("relation", "at least")
                if rel == "at least":
                    return n_sent >= target
                return n_sent <= target
            if instr_id.startswith("length_constraints:number_paragraphs"):
                n_para = len([p for p in text.split("\n\n") if p.strip()])
                return n_para == kwargs.get("num_paragraphs", 0)
            if instr_id.startswith("change_case:english_lowercase"):
                return text == text.lower()
            if instr_id.startswith("detectable_format:number_bullet_lists"):
                n_bullets = len(
                    re.findall(
                        r"^\s*[\*\-]\s",
                        text,
                        flags=re.MULTILINE))
                return n_bullets == kwargs.get("num_bullets", 0)
        except Exception:
            return False
        return True  # unsupported id slipped through -> don't penalize

    def grade(self, output_text, gold):
        ids = gold["instruction_id_list"]
        kwargs_list = gold["kwargs"]
        results = []
        for instr_id, kwargs in zip(ids, kwargs_list):
            if not instr_id.startswith(self._SUPPORTED_PREFIXES):
                continue
            results.append(
                self._check_one(
                    output_text,
                    instr_id,
                    kwargs or {}))
        return len(results) > 0 and all(results)


# ================================================================ medium tier
class MATHFullBenchmark(Benchmark):
    """Medium / Math — full MATH test set (documented: 72.1%).

    NOTE: the original `hendrycks/competition_math` repo was disabled on the
    Hub following a DMCA takedown. `DigitalLearningGmbH/MATH-lighteval` is a
    re-upload with the identical schema (problem/level/type/solution) and a
    `default` config that already concatenates all subjects into one
    5000-example test split, so no other code needs to change.
    """

    name = "MATH-full"
    difficulty_tier = "medium"
    task_type = "math"

    def load(self, n=30):
        ds = load_dataset(
            "DigitalLearningGmbH/MATH-lighteval",
            "default",
            split="test",
        )
        ds = ds.select(range(min(n, len(ds))))
        questions = []
        for row in ds:
            gold = _extract_boxed(row["solution"])
            if gold is None:
                continue
            questions.append({"question": row["problem"], "gold": gold})
        return questions

    def grade(self, output_text, gold):
        pred = _extract_boxed(output_text)
        if pred is None:
            pred = output_text.strip().splitlines()[-1].strip()
        return pred.replace(" ", "") == gold.replace(" ", "")


class MMLUProBenchmark(Benchmark):
    """Medium / General knowledge — MMLU-Pro (not documented for this model;
    treated as an exploratory tier-3 comparator).
    """

    name = "MMLU-Pro"
    difficulty_tier = "medium"
    task_type = "general_knowledge"

    def load(self, n=30):
        ds = load_dataset("TIGER-Lab/MMLU-Pro", split="test").select(range(n))
        questions = []
        for row in ds:
            options = row["options"]
            choices = "\n".join(
                f"{chr(65 + i)}. {c}" for i, c in enumerate(options))
            q = f"{row['question']}\n{choices}\nAnswer with just the letter."
            questions.append({"question": q, "gold": row["answer"]})
        return questions

    def grade(self, output_text, gold):
        match = re.search(r"\b([A-J])\b", output_text.strip())
        return match is not None and match.group(1) == gold


class BFCLv3Benchmark(Benchmark):
    """Medium / Agentic-tool-use — BFCL "simple" single-turn Python
    category (documented: 68.1, originally reported against BFCL v3).

    NOTE: the gorilla repo has since moved to BFCL v4 and split "simple" by
    language (BFCL_v3_simple.json doesn't exist anymore). This now points at
    BFCL_v4_simple_python.json, which has the same id/question/function
    schema and ground_truth format, so load()/grade() are unchanged. The
    question set itself may differ somewhat from the original v3 "simple"
    set (399 examples vs. the older count), so treat this as the current
    closest equivalent rather than a byte-identical rerun of v3.

    NOTE (simplification): the official BFCL grader does semantic AST
    matching against a possible-answer set including type/value constraints.
    This re-implementation does a lighter check: parse the model's function
    call syntactically, confirm the function name matches, and confirm every
    required argument's value is among the acceptable values. This is a
    reasonable approximation, not a drop-in replacement for the official
    checker (`bfcl.eval_checker`).
    """

    name = "BFCL-v3-simple"
    difficulty_tier = "medium"
    task_type = "agentic_tool_use"

    REPO_DIR = "/kaggle/working/gorilla"
    QUESTION_GLOB = "**/BFCL_v4_simple_python.json"
    ANSWER_GLOB = "**/possible_answer/BFCL_v4_simple_python.json"

    def _ensure_repo(self):
        if not os.path.exists(self.REPO_DIR):
            subprocess.run(
                ["git", "clone", "--depth", "1",
                 "https://github.com/ShishirPatil/gorilla.git", self.REPO_DIR],
                check=False,
            )

    def load(self, n=30):
        self._ensure_repo()
        q_files = glob.glob(
            os.path.join(
                self.REPO_DIR,
                self.QUESTION_GLOB),
            recursive=True)
        a_files = glob.glob(
            os.path.join(
                self.REPO_DIR,
                self.ANSWER_GLOB),
            recursive=True)
        if not q_files or not a_files:
            raise FileNotFoundError(
                "BFCL_v3_simple.json / possible_answer file not found under "
                f"{self.REPO_DIR} — the gorilla repo layout may have changed; "
                "update QUESTION_GLOB/ANSWER_GLOB."
            )

        def read_jsonl(path):
            with open(path) as f:
                return [json.loads(line) for line in f if line.strip()]

        questions_raw = read_jsonl(q_files[0])[:n]
        answers_raw = {row["id"]: row["ground_truth"]
                       for row in read_jsonl(a_files[0])}

        questions = []
        for row in questions_raw:
            qid = row["id"]
            if qid not in answers_raw:
                continue
            turns = row["question"][0] if isinstance(
                row["question"][0], list) else row["question"]
            user_msg = next(
                (t["content"] for t in turns if t["role"] == "user"),
                turns[0]["content"])
            funcs_desc = json.dumps(row["function"], indent=2)
            prompt = (
                f"You can call the following function(s):\n{funcs_desc}\n\n"
                f"User request: {user_msg}\n\n"
                "Respond with exactly one Python-style function call, "
                "e.g. func_name(arg1=value1, arg2=value2)."
            )
            questions.append({"question": prompt, "gold": answers_raw[qid]})
        return questions

    def grade(self, output_text, gold):
        # gold: list like [{"func_name": {"arg": ["acceptable", "values"]}}]
        match = re.search(
            r"(\w+)\s*\((.*)\)",
            output_text.strip(),
            flags=re.DOTALL)
        if not match:
            return False
        called_name, args_str = match.group(1), match.group(2)

        gold_entry = None
        for entry in gold:
            if called_name in entry:
                gold_entry = entry[called_name]
                break
        if gold_entry is None:
            return False

        try:
            call_ast = ast.parse(f"f({args_str})").body[0].value
            called_kwargs = {
                kw.arg: ast.literal_eval(
                    kw.value) for kw in call_ast.keywords}
        except Exception:
            return False

        for arg_name, acceptable in gold_entry.items():
            if arg_name not in called_kwargs:
                return False
            if str(called_kwargs[arg_name]) not in [str(v)
                                                    for v in acceptable]:
                return False
        return True


# ================================================================== hard tier
class AIME2025Benchmark(Benchmark):
    """Hard / Math — AIME 2025 instead of AIME 2024: lower ceiling for
    Qwen3-8B (documented: 63.3%) means more headroom to see compute-driven
    gains, and a more recent problem set lowers contamination risk.
    """

    name = "AIME2025"
    difficulty_tier = "hard"
    task_type = "math"

    def load(self, n=30):
        # NOTE: HuggingFaceH4/aime_2025 no longer resolves on the Hub.
        # test-time-compute/aime_2025 is the current replacement: single
        # "test" split, 30 problems (AIME 2025 I + II combined), fields
        # question/answer instead of problem/answer.
        ds = load_dataset("test-time-compute/aime_2025", split="test")
        ds = ds.select(range(min(n, 30, len(ds))))
        question_key = "question" if "question" in ds.column_names else "problem"
        questions = []
        for row in ds:
            questions.append({
                "question": row[question_key],
                "gold": str(row["answer"]).strip(),
            })
        return questions

    def grade(self, output_text, gold):
        numbers = re.findall(r"-?\d+", output_text)
        if not numbers:
            return False
        return numbers[-1].strip() == gold


class GPQADiamondBenchmark(Benchmark):
    """Hard / PhD-level science — GPQA-Diamond (documented: 62.0%).

    NOTE: this dataset is gated on HF — accept the terms at
    huggingface.co/datasets/Idavidrein/gpqa and run `huggingface-cli login`
    (or set HF_TOKEN) before calling .load(). The raw file always lists the
    correct answer first, so we deterministically shuffle per-question to
    avoid a positional-bias artifact.
    """

    name = "GPQA-Diamond"
    difficulty_tier = "hard"
    task_type = "phd_science"

    def load(self, n=30):
        ds = load_dataset("Idavidrein/gpqa", "gpqa_diamond", split="train")
        ds = ds.select(range(min(n, len(ds))))
        questions = []
        for i, row in enumerate(ds):
            options = [
                row["Correct Answer"],
                row["Incorrect Answer 1"],
                row["Incorrect Answer 2"],
                row["Incorrect Answer 3"],
            ]
            rng = random.Random(i)  # deterministic per-question shuffle
            order = list(range(4))
            rng.shuffle(order)
            shuffled = [options[j] for j in order]
            gold_letter = chr(65 + order.index(0))
            choices = "\n".join(
                f"{chr(65 + i)}. {c}" for i, c in enumerate(shuffled))
            q = f"{row['Question']}\n{choices}\nAnswer with just the letter."
            questions.append({"question": q, "gold": gold_letter})
        return questions

    def grade(self, output_text, gold):
        match = re.search(r"\b([A-D])\b", output_text.strip())
        return match is not None and match.group(1) == gold


class LiveCodeBenchV6Benchmark(Benchmark):
    """Hard / Coding — LiveCodeBench v6 (documented: 57.5%).

    NOTE (simplification): grades against `public_test_cases` only (stdin/
    stdout pairs), run in a subprocess with a timeout. `private_test_cases`
    (zlib+base64+pickle-encoded) are not decoded here; using only public
    cases is a weaker but common practical approximation, and yields an
    optimistic bound on true pass rate. A problem counts as correct only if
    every public test case passes.
    """

    name = "LiveCodeBench-v6"
    difficulty_tier = "hard"
    task_type = "coding"
    TIME_LIMIT_S = 6

    def load(self, n=30):
        # NOTE: livecodebench/code_generation_lite uses a Python loading
        # script (with a version_tag= kwarg), which recent `datasets`
        # versions refuse to run ("Dataset scripts are no longer
        # supported"). lighteval/code_generation_lite is a parquet mirror
        # with the identical row schema, using ordinary config names
        # (e.g. "release_v6") instead of a version_tag kwarg.
        ds = load_dataset(
            "lighteval/code_generation_lite",
            "release_v6",
            split="test",
        )
        ds = ds.select(range(min(n, len(ds))))
        questions = []
        for row in ds:
            try:
                tests = json.loads(row["public_test_cases"])
            except Exception:
                tests = []
            if not tests:
                continue
            prompt = (
                f"{row['question_content']}\n\n"
                f"Starter code (if any):\n{row.get('starter_code', '')}\n\n"
                "Write a complete Python solution that reads input from stdin "
                "and writes output to stdout. Return the code in a single "
                "```python fenced block."
            )
            questions.append({"question": prompt, "gold": tests})
        return questions

    @staticmethod
    def _extract_code(text):
        match = re.search(r"```python\s*(.*?)```", text, flags=re.DOTALL)
        if match:
            return match.group(1)
        match = re.search(r"```\s*(.*?)```", text, flags=re.DOTALL)
        return match.group(1) if match else text

    def grade(self, output_text, gold):
        code = self._extract_code(output_text)
        for case in gold:
            try:
                with tempfile.NamedTemporaryFile(
                    "w", suffix=".py", delete=False
                ) as f:
                    f.write(code)
                    path = f.name
                result = subprocess.run(
                    ["python3", path],
                    input=case.get("input", ""),
                    capture_output=True,
                    text=True,
                    timeout=self.TIME_LIMIT_S,
                )
                if result.stdout.strip() != case.get("output", "").strip():
                    return False
            except Exception:
                return False
            finally:
                try:
                    os.remove(path)
                except OSError:
                    pass
        return True

## 3. Raw data collector (Data Collection) — no accuracy/aggregation computed during the run

Collection protocol, as agreed:
- Every question is run at **every predefined compute level**.
- At each compute level, **≥5 samples** are taken (separate calls, temperature > 0).
- Every row is stored with its full label (`benchmark`, `difficulty_tier`, `task_type`) plus the raw call result.
- **No normalization or aggregation happens here** — that's done entirely in the analysis phase below, on the stored table.


In [87]:
import time
import pandas as pd
from openai import OpenAI


class ThinkingBudgetEvaluator:
    """Pure data-collection layer.

    Runs every question from every benchmark at every compute level, N samples
    each, and returns one row per single model call with full labels attached.
    No accuracy/aggregation is computed here by design — see the analysis
    section below, which works entirely off the raw table.
    """

    def __init__(self, server: VLLMServer):
        self.server = server
        self.client = OpenAI(
            base_url=f"http://localhost:{server.port}/v1", api_key="dummy")

    def query(self, question, budget, max_tokens, temperature=0.7):
        t0 = time.time()
        response = self.client.chat.completions.create(
            model=self.server.model,
            messages=[{"role": "user", "content": question}],
            max_tokens=max_tokens,
            temperature=temperature,
            extra_body={"vllm_xargs": {"max_thinking_tokens": budget}},
        )
        elapsed = time.time() - t0
        full_text = response.choices[0].message.content or ""
        tokens_used = response.usage.completion_tokens

        # Split thinking vs final output instead of discarding the thinking part
        if "<think>" in full_text and "</think>" in full_text:
            # Normal case: closed cleanly
            thinking = full_text.split("<think>")[1].split("</think>")[0].strip()
            output = full_text.split("</think>")[1].strip()
            was_truncated = False
        elif "<think>" in full_text:
            # Hit max_tokens while still inside <think> -- capture the partial
            # reasoning instead of losing it, and flag the row as truncated.
            thinking = full_text.split("<think>")[1].strip()
            output = ""  # no real answer was produced, it got cut off mid-thought
            was_truncated = True
        else:
            thinking = ""
            output = full_text.strip()
            was_truncated = (tokens_used >= max_tokens)

        return output, thinking, tokens_used, elapsed, was_truncated

    def collect(self, benchmarks, compute_levels, n_questions=30,
                n_samples=5, max_tokens=2000, temperature=0.7, answer_margin=800):
        """benchmarks: list of Benchmark instances -- mixing tiers here
        is fine, since every row keeps its own benchmark/tier/task_type label.

        answer_margin: extra tokens reserved on top of each budget so the
        model has room to close </think> and still write a real answer.
        The effective cap per call is max(max_tokens, budget + answer_margin) --
        without this, a large budget with a small fixed max_tokens gets cut
        off mid-thinking and never produces an answer at all.

        Returns a raw, unaggregated DataFrame: one row per single model call.
        """
        rows = []

        for bench in benchmarks:
            questions = bench.load(n_questions)

            for qi, q in enumerate(questions):
                question_id = f"{bench.name}_{qi}"

                for budget in compute_levels:
                    effective_max_tokens = max(max_tokens, budget + answer_margin)

                    for sample_idx in range(n_samples):
                        output, thinking, tokens, elapsed, was_truncated = self.query(
                            q["question"], budget, effective_max_tokens, temperature
                        )
                        is_correct = bench.grade(output, q["gold"])

                        rows.append({
                            "question_id": question_id,
                            "benchmark": bench.name,
                            "difficulty_tier": bench.difficulty_tier,
                            "task_type": bench.task_type,
                            "compute_level": budget,
                            "sample_index": sample_idx,
                            "raw_model_output": output,
                            "thinking_text": thinking,
                            "gold": q["gold"],
                            "is_correct": is_correct,
                            "tokens_used": tokens,
                            "latency_s": elapsed,
                            "was_truncated": was_truncated,
                        })

                        print(
                            f"[collect] {bench.name} ({bench.difficulty_tier}) "
                            f"q={qi + 1}/{len(questions)} budget={budget} "
                            f"sample={sample_idx + 1}/{n_samples} "
                            f"correct={is_correct} tokens={tokens} "
                            f"truncated={was_truncated}",
                            flush=True,
                        )

        return pd.DataFrame(rows)

## 4. Pilot, then the full compute-level sweep

Per the agreed execution approach:
1. **Pilot** (5–10 questions, 2 widely-spaced compute levels, 2 samples): sanity-check that the parser/evaluator works for every `task_type`, that the thinking-budget actually changes model behavior, and that no tier is trivially stuck at 0%/100%.
2. **Full run**: once the pilot looks correct, sweep every compute level (`256, 512, 1024, 2048, 4096, 8192`) at 10 questions/benchmark, ≥5 samples each, logging every single-call row into the raw table — no aggregation yet.

### 4.1 Pilot run

In [89]:
benchmarks = [
    GSM8KBenchmark(),            # easy / math
    MMLURedux(),                 # easy / general_knowledge
    IFEvalBenchmark(),           # easy / instruction_following
    MATHFullBenchmark(),         # medium / math
    MMLUProBenchmark(),          # medium / general_knowledge
    BFCLv3Benchmark(),           # medium / agentic_tool_use
    AIME2025Benchmark(),         # hard / math
    GPQADiamondBenchmark(),      # hard / phd_science
    LiveCodeBenchV6Benchmark(),  # hard / coding
]

PILOT_N_QUESTIONS = 1
PILOT_N_SAMPLES = 1
# Two levels far apart, not the full sweep: enough to confirm the budget
# actually changes model behavior (token usage, and ideally accuracy) before
# paying for the full 6-level x 10-question x 5-sample run below.
PILOT_COMPUTE_LEVELS = [256, 8192]

evaluator = ThinkingBudgetEvaluator(server)

pilot_df = evaluator.collect(
    benchmarks=benchmarks,
    compute_levels=PILOT_COMPUTE_LEVELS,
    n_questions=PILOT_N_QUESTIONS,
    n_samples=PILOT_N_SAMPLES,
)

pilot_df.to_csv("/kaggle/working/h1_pilot_results.csv", index=False)

print("\n" + "=" * 60)
print("[pilot] SANITY CHECKS")
print("=" * 60)

# 1) parser/evaluator sanity — every (benchmark, task_type) produced rows
seen = pilot_df.groupby(["benchmark", "difficulty_tier", "task_type"]).size()
expected = {(b.name, b.difficulty_tier, b.task_type) for b in benchmarks}
missing = expected - set(seen.index)
if missing:
    print(
        f"[pilot] ⚠ no rows collected for: {missing} — "
        "check load()/grade() for these"
    )
else:
    print("[pilot] ✓ every benchmark produced rows")

# 2) budget-enforcement sanity — completion tokens should generally track the
#    requested thinking budget (low budget -> low tokens_used, roughly)
budget_effect = (
    pilot_df.groupby(["benchmark", "compute_level"])["tokens_used"]
    .mean()
    .unstack("compute_level")
)
print(
    "\n[pilot] mean completion tokens by compute_level "
    "(should increase left->right):"
)
print(budget_effect)

# 3) no tier trivially at the accuracy floor/ceiling
tier_acc = pilot_df.groupby("difficulty_tier")["is_correct"].mean()
print("\n[pilot] accuracy by tier (flagging anything at/near 0% or 100%):")
for tier, acc in tier_acc.items():
    flag = "  <-- CHECK THIS" if acc <= 0.0 or acc >= 1.0 else ""
    print(f"  {tier}: {acc:.1%}{flag}")

bench_acc = pilot_df.groupby(["benchmark", "difficulty_tier", "task_type"])[
    "is_correct"].mean()
print("\n[pilot] accuracy by benchmark:")
print(bench_acc)

[collect] GSM8K (easy) q=1/1 budget=256 sample=1/1 correct=True tokens=388 truncated=False
[collect] GSM8K (easy) q=1/1 budget=8192 sample=1/1 correct=True tokens=1191 truncated=False
[collect] MMLU-Redux (easy) q=1/1 budget=256 sample=1/1 correct=True tokens=262 truncated=False
[collect] MMLU-Redux (easy) q=1/1 budget=8192 sample=1/1 correct=True tokens=341 truncated=False
[collect] IFEval (easy) q=1/1 budget=256 sample=1/1 correct=True tokens=731 truncated=False
[collect] IFEval (easy) q=1/1 budget=8192 sample=1/1 correct=True tokens=1085 truncated=False
[collect] MATH-full (medium) q=1/1 budget=256 sample=1/1 correct=True tokens=614 truncated=False
[collect] MATH-full (medium) q=1/1 budget=8192 sample=1/1 correct=True tokens=1813 truncated=False
[collect] MMLU-Pro (medium) q=1/1 budget=256 sample=1/1 correct=True tokens=278 truncated=False
[collect] MMLU-Pro (medium) q=1/1 budget=8192 sample=1/1 correct=False tokens=1674 truncated=False
[collect] BFCL-v3-simple (medium) q=1/1 budget

### 4.2 Full run across all compute levels

In [90]:
pilot_df["thinking_words"] = pilot_df["thinking_text"].fillna("").str.split().str.len()
pilot_df

,question_id,benchmark,difficulty_tier,task_type,compute_level,sample_index,raw_model_output,thinking_text,gold,is_correct,tokens_used,latency_s,was_truncated,thinking_words
0,GSM8K_0,GSM8K,easy,math,256,0,Janet’s ducks lay 16 eggs per day. She uses 3 ...,"Okay, let's see. Janet has ducks that lay 16 e...",18,True,388,13.842318,False,190
1,GSM8K_0,GSM8K,easy,math,8192,0,Janet's ducks lay 16 eggs per day. She uses 3 ...,"Okay, so Janet has these ducks that lay 16 egg...",18,True,1191,44.091401,False,747
2,MMLU-Redux_0,MMLU-Redux,easy,general_knowledge,256,0,B,"Okay, let's tackle this question. The question...",B,True,262,8.934947,False,198
3,MMLU-Redux_0,MMLU-Redux,easy,general_knowledge,8192,0,B,"Okay, let's see. The question is asking which ...",B,True,341,11.719803,False,266
4,IFEval_0,IFEval,easy,instruction_following,256,0,Raymond III was a prominent Crusader leader wh...,"Okay, the user wants a 300+ word summary of th...",{'instruction_id_list': ['punctuation:no_comma...,True,731,25.789557,False,195
5,IFEval_0,IFEval,easy,instruction_following,8192,0,Raymond III was born in 1155 and died in 1194....,"Okay, the user wants a 300+ word summary of th...",{'instruction_id_list': ['punctuation:no_comma...,True,1085,38.701506,False,456
6,MATH-full_0,MATH-full,medium,math,256,0,To determine how many vertical asymptotes the ...,"Okay, so I need to figure out how many vertica...",2,True,614,21.363265,False,186
7,MATH-full_0,MATH-full,medium,math,8192,0,To determine how many vertical asymptotes the ...,"Okay, so I need to figure out how many vertica...",2,True,1813,65.659901,False,880
8,MMLU-Pro_0,MMLU-Pro,medium,general_knowledge,256,0,"I. Unsafe practices, Distress, Fear, Serious\n...","Okay, let's tackle this question. The user is ...",I,True,278,9.659728,False,183
9,MMLU-Pro_0,MMLU-Pro,medium,general_knowledge,8192,0,The question asks about typical advertising re...,"Okay, let's tackle this question. So, the ques...",I,False,1674,60.618670,False,969


In [ ]:
COMPUTE_LEVELS = [256, 512, 1024, 2048, 4096, 8192]
N_QUESTIONS_PER_BENCHMARK = 10   # full-run scale, per the agreed plan
N_SAMPLES = 5                    # agreed minimum per compute level

evaluator = ThinkingBudgetEvaluator(server)

raw_df = evaluator.collect(
    benchmarks=benchmarks,          # reuse the 9-benchmark set defined above
    compute_levels=COMPUTE_LEVELS,
    n_questions=N_QUESTIONS_PER_BENCHMARK,
    n_samples=N_SAMPLES,
)

raw_df.to_csv("/kaggle/working/h1_raw_results.csv", index=False)
raw_df.head()

## 5. Analysis phase — entirely queries on the raw table (no re-running any calls)

### 5.1 Separate curve for each (benchmark × tier) first


In [ ]:
raw_df = pd.read_csv("/kaggle/working/h1_raw_results.csv")

curve_df = (
    raw_df
    .groupby(["benchmark", "difficulty_tier", "compute_level"])
    .agg(accuracy=("is_correct", "mean"), n=("is_correct", "size"))
    .reset_index()
    .sort_values(["benchmark", "compute_level"])
)
curve_df


### 5.2 Normalize each curve, then aggregate (avoiding Simpson's paradox)

- Each curve (benchmark × tier) is normalized on its own (min-max) based on its performance across compute levels.
- The aggregate curve = mean of the normalized curves, not the mean of raw accuracy.
- A per-tier mean (normalized) is also computed, for use in the knee analysis below.


In [ ]:
import numpy as np


def normalize(series):
    s = series.values.astype(float)
    if s.max() == s.min():
        return np.zeros_like(s)
    return (s - s.min()) / (s.max() - s.min())


curve_df["accuracy_norm"] = (
    curve_df.groupby(["benchmark", "difficulty_tier"])["accuracy"]
    .transform(normalize)
)

# aggregate = mean of normalized curves across ALL (benchmark, tier) groups
aggregate_df = (
    curve_df.groupby("compute_level")["accuracy_norm"]
    .mean()
    .reset_index()
    .rename(columns={"accuracy_norm": "aggregate_accuracy_norm"})
    .sort_values("compute_level")
)

# per-tier normalized curve (average across benchmarks within the same tier)
tier_df = (
    curve_df.groupby(["difficulty_tier", "compute_level"])["accuracy_norm"]
    .mean()
    .reset_index()
    .rename(columns={"accuracy_norm": "tier_accuracy_norm"})
    .sort_values(["difficulty_tier", "compute_level"])
)

aggregate_df, tier_df


### 5.3 Plot disaggregated alongside the aggregate (robustness / sanity check)


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for (bench, tier), g in curve_df.groupby(["benchmark", "difficulty_tier"]):
    axes[0].plot(
        g["compute_level"],
        g["accuracy"],
        marker="o",
        label=f"{bench} ({tier})")
axes[0].set_title("Disaggregated — raw accuracy per benchmark")
axes[0].set_xlabel("compute level (thinking budget)")
axes[0].set_ylabel("accuracy")
axes[0].legend(fontsize=8)

axes[1].plot(
    aggregate_df["compute_level"],
    aggregate_df["aggregate_accuracy_norm"],
    marker="o", color="black", linewidth=2.5,
    label="aggregate (mean of normalized curves)",
)
for tier, g in tier_df.groupby("difficulty_tier"):
    axes[1].plot(
        g["compute_level"],
        g["tier_accuracy_norm"],
        marker="o",
        linestyle="--",
        label=f"{tier} (normalized)")
axes[1].set_title("Aggregate vs per-tier — normalized")
axes[1].set_xlabel("compute level (thinking budget)")
axes[1].set_ylabel("normalized accuracy")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

### 5.4 Knee detection (kneedle) and Δcompute per tier

- Knee of the aggregate curve → **global knee**.
- Separate knee for each tier (easy/medium/hard) → **tier-specific knee**.
- `Δcompute = compute at global knee − compute at tier knee`:
  - Large positive value for easy → quantified evidence of **overthinking**.
  - Negative value or no knee detected for hard → quantified evidence of **underthinking**.


In [ ]:
!pip install -q kneed
from kneed import KneeLocator


def find_knee(x, y):
    try:
        kl = KneeLocator(x, y, curve="concave", direction="increasing")
        return kl.knee
    except Exception:
        return None


global_knee = find_knee(
    aggregate_df["compute_level"].tolist(),
    aggregate_df["aggregate_accuracy_norm"].tolist(),
)
print(f"[knee] global knee at compute_level = {global_knee}")

tier_knees = {}
for tier, g in tier_df.groupby("difficulty_tier"):
    g_sorted = g.sort_values("compute_level")
    knee = find_knee(
        g_sorted["compute_level"].tolist(),
        g_sorted["tier_accuracy_norm"].tolist())
    tier_knees[tier] = knee
    print(f"[knee] {tier} tier knee at compute_level = {knee}")

print(
    "\n[delta] Δcompute = global_knee - tier_knee "
    "(positive => overthinking signal, "
    "negative/None => underthinking signal)"
)
for tier, knee in tier_knees.items():
    if knee is None or global_knee is None:
        print(f"  {tier}: knee not detected — inspect the curve manually")
    else:
        print(f"  {tier}: Δcompute = {global_knee - knee}")

### 5.5 (Optional) Visual confirmation: test varying difficulty at the same global-knee compute level

An additional confirmatory step after computing the knees — not a substitute for them.


In [ ]:
if global_knee is not None:
    at_knee = raw_df[raw_df["compute_level"] == global_knee]
    summary = (
        at_knee.groupby("difficulty_tier")["is_correct"]
        .mean()
        .reset_index()
        .rename(columns={"is_correct": "accuracy_at_global_knee"})
    )
    print(
        "accuracy per difficulty tier at global knee "
        f"(compute_level={global_knee}):"
    )
    summary
else:
    print(
        "Global knee not detected — widen COMPUTE_LEVELS "
        "or inspect the aggregate curve."
    )

### 5.6 Save deliverables

Persist the final curves (raw per-subgroup, normalized per-subgroup, normalized per-tier, aggregate) and the knee-detection outputs (global knee, tier-specific knees, Δcompute per tier) as the output of the analysis phase.

In [ ]:
import json as _json

OUT_DIR = "/kaggle/working/h1_analysis_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# raw per-subgroup curve (accuracy + n), normalized per-subgroup curve
curve_df.to_csv(os.path.join(OUT_DIR, "curves_per_subgroup.csv"), index=False)

# per-tier normalized curve
tier_df.to_csv(
    os.path.join(
        OUT_DIR,
        "curves_per_tier_normalized.csv"),
    index=False)

# aggregate normalized curve
aggregate_df.to_csv(
    os.path.join(
        OUT_DIR,
        "curve_aggregate_normalized.csv"),
    index=False)

# knee-detection outputs: global knee, tier-specific knees, delta-compute
# per tier
knee_results = {
    "global_knee": global_knee,
    "tier_knees": tier_knees,
    "delta_compute_per_tier": {
        tier: (None if (knee is None or global_knee is None)
               else global_knee - knee)
        for tier, knee in tier_knees.items()
    },
}
with open(os.path.join(OUT_DIR, "knee_results.json"), "w") as f:
    _json.dump(knee_results, f, indent=2)

print(f"[deliverables] saved to {OUT_DIR}:")
for fn in sorted(os.listdir(OUT_DIR)):
    print(f"  - {fn}")